In [ ]:
# 1
# Check GPU.
!nvidia-smi

Fri Jun  5 06:00:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   27C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 2
# Install dependencies.
!pip -q install -U uv

# Basic dependencies.
!uv pip install --system -U openai tqdm requests psutil pandas

# Install recent vLLM nightly for CUDA 13.0 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

# Version check.
import sys
import torch
import vllm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)

Using Python 3.12.13 environment at: /usr
Resolved 24 packages in 80ms
Prepared 1 package in 0.25ms
Uninstalled 1 package in 8ms
Installed 1 package in 12ms
 - numpy==2.3.5
 + numpy==2.4.6
Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 7.88s
Prepared 1 package in 0.26ms
Uninstalled 1 package in 7ms
Installed 1 package in 12ms
 - numpy==2.4.6
 + numpy==2.3.5
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
vLLM: 0.22.1rc1.dev196+g96229fa99


In [ ]:
# 3
# Mount Google Drive and prepare paths.
from google.colab import drive
from pathlib import Path
import shutil
import json
import os
import re

drive.mount("/content/drive")

# Define input files.
MODEL_DATASET_FILES = {
    "qwen3.5": {
        "hotpotqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "hotpot_answers_qwen3_5_9b/qwen_agent_responses.json"
        ),
        "2wikimultihopqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "wiki_answers_qwen3_5_9b/qwen_agent_responses.json"
        ),
    },
    "gemma4": {
        "hotpotqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "hotpot_answers_gemma4/qwen_agent_responses.json"
        ),
        "2wikimultihopqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "wiki_answers_gemma4/qwen_agent_responses.json"
        ),
    },
    "gpt-oss-120b": {
        "hotpotqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "hotpot_answers_gpt_oss_120b/qwen_agent_responses.json"
        ),
        "2wikimultihopqa": Path(
            "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
            "wiki_answers_gpt_oss_120b/qwen_agent_responses.json"
        ),
    },
}

LOCAL_WORK_DIR = Path("/content/kg_llm_judge")
LOCAL_INPUT_DIR = LOCAL_WORK_DIR / "inputs"
LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

GDRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/"
    "llm_judge_qwen35_27b"
)
GDRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def safe_name(text: str) -> str:
    # Make safe file name.
    text = str(text).strip().replace(".", "_")
    return re.sub(r"[^A-Za-z0-9_-]+", "_", text)

JUDGE_TASKS = []

# Copy files to local disk.
for answer_model_name, dataset_map in MODEL_DATASET_FILES.items():
    for dataset_name, src in dataset_map.items():
        assert src.exists(), f"Input file not found: {src}"

        task_id = f"{safe_name(answer_model_name)}__{safe_name(dataset_name)}"
        dst = LOCAL_INPUT_DIR / f"{task_id}.json"

        shutil.copy2(src, dst)

        task = {
            "task_id": task_id,
            "answer_model": answer_model_name,
            "dataset": dataset_name,
            "source_path": src,
            "local_path": dst,
        }

        JUDGE_TASKS.append(task)

        print("Copied:", src)
        print("     ->", dst)

print("\nTasks:", len(JUDGE_TASKS))
print("Local input dir:", LOCAL_INPUT_DIR)
print("Output dir:", GDRIVE_OUTPUT_DIR)

for task in JUDGE_TASKS:
    print(task["task_id"], "|", task["answer_model"], "|", task["dataset"])

Mounted at /content/drive
Copied: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_qwen3_5_9b/qwen_agent_responses.json
     -> /content/kg_llm_judge/inputs/qwen3_5__hotpotqa.json
Copied: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_qwen3_5_9b/qwen_agent_responses.json
     -> /content/kg_llm_judge/inputs/qwen3_5__2wikimultihopqa.json
Copied: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gemma4/qwen_agent_responses.json
     -> /content/kg_llm_judge/inputs/gemma4__hotpotqa.json
Copied: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gemma4/qwen_agent_responses.json
     -> /content/kg_llm_judge/inputs/gemma4__2wikimultihopqa.json
Copied: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b/qwen_agent_responses.json
     -> /content/kg_llm_judge/inputs/gpt-oss-120b__hotpotqa.json
Copied: /content/drive/MyDrive/final_project/baseline/DAT

In [ ]:
# 4
# Load input JSON files.
from collections import Counter

def load_json_list(path: Path):
    # Load a JSON list.
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    assert isinstance(data, list), f"Expected a list in {path}"
    return data

all_input_data = {}

required_keys = {"type", "question", "gt", "response"}

for task in JUDGE_TASKS:
    task_id = task["task_id"]
    path = task["local_path"]

    data = load_json_list(path)
    all_input_data[task_id] = data

    print("\nTask:", task_id)
    print("Answer model:", task["answer_model"])
    print("Dataset:", task["dataset"])
    print("Rows:", len(data))
    print("Types:", Counter(x.get("type") for x in data))

    if len(data) > 0:
        keys = set(data[0].keys())
        print("Keys:", sorted(keys))
        missing = required_keys - keys
        print("Missing required keys:", sorted(missing))

    for i, row in enumerate(data[:5]):
        missing = required_keys - set(row.keys())
        assert not missing, f"Missing keys in {task_id}, row {i}: {missing}"


Task: qwen3_5__hotpotqa
Answer model: qwen3.5
Dataset: hotpotqa
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['gt', 'question', 'response', 'type']
Missing required keys: []

Task: qwen3_5__2wikimultihopqa
Answer model: qwen3.5
Dataset: 2wikimultihopqa
Rows: 1000
Types: Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})
Keys: ['gt', 'question', 'response', 'type']
Missing required keys: []

Task: gemma4__hotpotqa
Answer model: gemma4
Dataset: hotpotqa
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['gt', 'question', 'response', 'type']
Missing required keys: []

Task: gemma4__2wikimultihopqa
Answer model: gemma4
Dataset: 2wikimultihopqa
Rows: 1000
Types: Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})
Keys: ['gt', 'question', 'response', 'type']
Missing required keys: []

Task: gpt-oss-120b__hotpotqa
Answer model: gpt-oss-120b
Dataset: hotpotqa
Rows: 1

In [ ]:
# 5
# Start vLLM server in Qwen non-thinking mode.
import subprocess
import time
import requests
import shlex
import psutil
from pathlib import Path
import os

MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Total context length.
MAX_MODEL_LEN = 12288

# Memory margin for RTX PRO 6000 Blackwell 96GB.
GPU_MEMORY_UTILIZATION = 0.92

# Throughput settings.
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

def kill_process_tree(pid):
    # Kill process tree.
    try:
        parent = psutil.Process(int(pid))

        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass

        parent.kill()
        parent.wait(timeout=10)
        print("Killed old process tree:", pid)

    except Exception:
        pass

# Stop old PID.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Kill leftover vLLM servers.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode.
    "--language-model-only",

    # Qwen non-thinking mode.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    # Throughput settings.
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    # Shared prompt prefix cache.
    "--enable-prefix-caching",

    # Use vLLM generation config.
    "--generation-config", "vllm",

    # Good dtype for Blackwell.
    "--dtype", "bfloat16",

    # Safe for custom model code.
    "--trust-remote-code",
]

server_env = os.environ.copy()

# Avoid FlashInfer sampler crash.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# CUDA 13.0 / Blackwell hints.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 12288 --gpu-memory-utilization 0.92 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 8 --max-num-batched-tokens 16384 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 2768
Log: /content/vllm_server.log


In [ ]:
# 6
# Wait for vLLM server.
import time
import requests
from pathlib import Path

def tail_log(path, n=80):
    # Return recent log lines.
    path = Path(path)

    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()

    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        health = requests.get(f"http://localhost:{PORT}/health", timeout=5)

        if health.status_code == 200:
            models = requests.get(f"{BASE_URL}/models", timeout=10)

            if models.status_code == 200:
                ready = True
                model_info = models.json()["data"][0]

                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break

    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")

        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)

        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=3315) INFO 06-05 06:03:30 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=3315) INFO 06-05 06:03:30 [parallel_state.py:1568] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:50073 backend=nccl
(EngineCore pid=3315) INFO 06-05 06:03:30 [parallel_state.py:1903] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=3315) INFO 06-05 06:03:31 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=3315) INFO 06-05 06:03:31 [gpu_model_runner.py:5087] Starting to load model Qwen/Qwen3.5-27B...
(EngineCore pid=3315) INFO 06-05 06:03:31 [cuda.py:433] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=

In [ ]:
# 7
# Define judge prompt and JSON schema.
JUDGE_SYSTEM_PROMPT = """
You are a strict but fair answer judge.

You will receive:
- a question
- the ground-truth answer
- a candidate LLM response

Your job is to judge whether the candidate response correctly answers the question.
Use the ground-truth answer as the authoritative answer.
Do not solve the question yourself.
Do not add background knowledge.
Do not write analysis outside JSON.

Return only this JSON object, with reason first and score second:
{
  "reason": "short reason",
  "score": 0 or 1
}

Rules:
- Score 1 if the candidate gives the correct answer to the question.
- Score 1 if the candidate is semantically the same as the ground truth, even with different wording.
- Score 1 if the candidate gives the main requested answer clearly and any missing part is only a harmless clarifying suffix.
- Score 1 if the candidate includes extra information that does not contradict the correct answer.
- Score 0 if the candidate is wrong.
- Score 0 if the candidate is incomplete for what the question asks.
- Score 0 if the candidate is vague, too broad, or too narrow.
- Score 0 if the candidate says the answer is unknown, unavailable, cannot be determined, or similar.
- Score 0 if the candidate contains a contradiction or an incorrect final answer.
- Use the question to decide what details are required.
- Do not require exact string matching.
- When unsure, score 0.

Keep the reason short: maximum 25 words.
""".strip()

JUDGE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "reason": {
            "type": "string"
        },
        "score": {
            "type": "integer",
            "enum": [0, 1]
        },
    },
    "required": ["reason", "score"],
    "additionalProperties": False,
}

def build_judge_user_prompt(question: str, gt: str, response: str) -> str:
    # Build judge prompt.
    return f"""
Question:
{question}

Ground-truth answer:
{gt}

Candidate LLM response:
{response}

Judge the candidate response.
Return only JSON with reason first and score second.
""".strip()

In [ ]:
# 8
# Create client and robust parser.
from openai import OpenAI
from typing import Any, Dict
import json
import time
import re

client = OpenAI(
    base_url=BASE_URL,
    api_key="EMPTY",
)

def extract_first_json_object(text: str) -> Dict[str, Any]:
    # Extract first JSON object.
    text = (text or "").strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")

    if start == -1:
        raise ValueError(f"No JSON object found: {text[:300]}")

    depth = 0
    in_str = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1

                if depth == 0:
                    return json.loads(text[start:i + 1])

    raise ValueError(f"Incomplete JSON object: {text[:300]}")

def clean_reason(reason: Any) -> str:
    # Clean reason text.
    reason = str(reason or "").strip()
    reason = reason.replace("\n", " ")
    reason = reason.replace('"', "'")
    reason = " ".join(reason.split())

    if not reason:
        reason = "No reason provided."

    return reason

def normalize_judge_output(obj: Dict[str, Any]) -> Dict[str, Any]:
    # Normalize judge output.
    score = obj.get("score")

    if isinstance(score, str):
        score = score.strip()
        if score in {"0", "1"}:
            score = int(score)

    if score not in {0, 1}:
        raise ValueError(f"Invalid score: {obj}")

    reason = clean_reason(obj.get("reason", ""))

    return {
        "reason": reason,
        "score": int(score),
        "parse_method": "json",
    }

def parse_malformed_json_like_text(text: str) -> Dict[str, Any]:
    # Recover malformed JSON-like text.
    text = (text or "").strip()

    score_match = re.search(r'(?is)"score"\s*:\s*([01])\b', text)
    if score_match is None:
        score_match = re.search(r"(?is)\bscore\s*[:=]\s*([01])\b", text)

    if score_match is None:
        raise ValueError(f"Could not recover score: {text[:300]}")

    score = int(score_match.group(1))

    reason = None

    # Case: reason before score.
    m = re.search(
        r'(?is)"reason"\s*:\s*"(.*)"\s*,\s*"score"\s*:\s*[01]',
        text,
    )
    if m is not None:
        reason = m.group(1)

    # Case: score before reason.
    if reason is None:
        m = re.search(
            r'(?is)"score"\s*:\s*[01]\s*,\s*"reason"\s*:\s*"(.*)"\s*\}?',
            text,
        )
        if m is not None:
            reason = m.group(1)

    # Plain text fallback.
    if reason is None:
        m = re.search(r"(?im)^\s*reason\s*:\s*(.+?)\s*$", text)
        if m is not None:
            reason = m.group(1)

    if reason is None:
        reason = "Recovered score from malformed judge output."

    return {
        "reason": clean_reason(reason),
        "score": score,
        "parse_method": "regex_recovered",
    }

def parse_judge_text(text: str) -> Dict[str, Any]:
    # Parse judge output.
    try:
        obj = extract_first_json_object(text)
        return normalize_judge_output(obj)
    except Exception:
        return parse_malformed_json_like_text(text)

In [ ]:
# 9
# Define one judge call with retries.
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 512
MAX_RETRIES = 5

REPAIR_MAX_RETRIES = 3
REPAIR_MAX_TOKENS = 128
REPAIR_TEMPERATURE = 0.0

REPAIR_SYSTEM_PROMPT = """
You are a strict but fair answer judge.

You will receive:
- a question
- the ground-truth answer
- a candidate LLM response

Judge whether the candidate response correctly answers the question.
Use the ground-truth answer as the authoritative answer.
Do not solve the question yourself.

Return exactly two lines:
reason: short reason without quotation marks
score: 0 or 1

Rules:
- Score 1 if the candidate gives the correct answer.
- Score 1 if the candidate is semantically the same as the ground truth.
- Score 1 if extra information is present but not contradictory.
- Score 1 if the candidate gives the main requested answer clearly and any missing part is only a harmless clarifying suffix.
- Score 0 if the candidate is wrong, incomplete, vague, too broad, or too narrow.
- Score 0 if the candidate says unknown, unavailable, cannot be determined, or similar.
- Score 0 if there is contradiction or an incorrect final answer.
- When unsure, score 0.
""".strip()

def build_repair_prompt(question: str, gt: str, response: str) -> str:
    # Build repair prompt.
    return f"""
Question:
{question}

Ground-truth answer:
{gt}

Candidate LLM response:
{response}

Return exactly:
reason: ...
score: 0 or 1
""".strip()

def parse_repair_output(text: str) -> Dict[str, Any]:
    # Parse repair output.
    text = (text or "").strip()

    score_match = re.search(r"(?im)^\s*score\s*:\s*([01])\s*$", text)

    if score_match is None:
        score_match = re.search(r"(?i)\bscore\s*[:=]\s*([01])\b", text)

    if score_match is None:
        raise ValueError(f"Could not parse score from: {text[:300]}")

    score = int(score_match.group(1))

    reason_match = re.search(r"(?im)^\s*reason\s*:\s*(.+?)\s*$", text)
    reason = reason_match.group(1).strip() if reason_match else "Parsed score from repair judge."

    return {
        "reason": clean_reason(reason),
        "score": score,
        "parse_method": "repair_text",
    }

def call_chat_completion(messages, mode: str):
    # Call vLLM endpoint.
    if mode == "guided_json":
        return client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=JUDGE_TEMPERATURE,
            max_tokens=JUDGE_MAX_TOKENS,
            extra_body={
                "guided_json": JUDGE_JSON_SCHEMA,
            },
        )

    if mode == "json_object":
        return client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=JUDGE_TEMPERATURE,
            max_tokens=JUDGE_MAX_TOKENS,
            response_format={"type": "json_object"},
        )

    raise ValueError(f"Unknown mode: {mode}")

def repair_judge_one_record(record: Dict[str, Any]) -> Dict[str, Any]:
    # Repair one failed row.
    question = str(record.get("question", ""))
    gt = str(record.get("gt", ""))
    response = str(record.get("response", ""))

    messages = [
        {
            "role": "system",
            "content": REPAIR_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_repair_prompt(question, gt, response),
        },
    ]

    last_error = None
    started = time.perf_counter()

    for attempt in range(REPAIR_MAX_RETRIES):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=REPAIR_TEMPERATURE,
                max_tokens=REPAIR_MAX_TOKENS,
            )

            raw_text = completion.choices[0].message.content or ""
            parsed = parse_repair_output(raw_text)

            return {
                "judge_reason": parsed["reason"],
                "judge_score": parsed["score"],
                "judge_raw": raw_text,
                "judge_error": None,
                "judge_latency_sec": time.perf_counter() - started,
                "judge_method": parsed["parse_method"],
            }

        except Exception as e:
            last_error = repr(e)
            time.sleep(1.5 * (attempt + 1))

    return {
        "judge_reason": "Repair failed after retries; defaulted to incorrect.",
        "judge_score": 0,
        "judge_raw": None,
        "judge_error": None,
        "judge_latency_sec": time.perf_counter() - started,
        "judge_method": "final_default_0",
        "judge_repair_error": last_error,
    }

def judge_one_record(record: Dict[str, Any]) -> Dict[str, Any]:
    # Judge one row.
    question = str(record.get("question", ""))
    gt = str(record.get("gt", ""))
    response = str(record.get("response", ""))

    messages = [
        {
            "role": "system",
            "content": JUDGE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_judge_user_prompt(question, gt, response),
        },
    ]

    last_error = None
    started = time.perf_counter()

    for attempt in range(MAX_RETRIES):
        for mode in ["guided_json", "json_object"]:
            try:
                completion = call_chat_completion(messages, mode=mode)

                raw_text = completion.choices[0].message.content or ""
                parsed = parse_judge_text(raw_text)

                return {
                    "judge_reason": parsed["reason"],
                    "judge_score": parsed["score"],
                    "judge_raw": raw_text,
                    "judge_error": None,
                    "judge_latency_sec": time.perf_counter() - started,
                    "judge_method": mode if parsed["parse_method"] == "json" else parsed["parse_method"],
                }

            except Exception as e:
                last_error = repr(e)

        time.sleep(2.0 * (attempt + 1))

    repaired = repair_judge_one_record(record)
    repaired["judge_primary_error"] = last_error
    return repaired

In [ ]:
# 10
# Define task judging with resume support.
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict, Any
import json

REQUEST_CONCURRENCY = 16
TASK_MAX_ROUNDS = 5

def make_result_key(record: Dict[str, Any], row_id: int) -> str:
    # Make stable key.
    if record.get("source_index") is not None:
        return f"source_index:{record.get('source_index')}"

    return f"row_id:{row_id}"

def is_valid_judgement(obj: Dict[str, Any] | None) -> bool:
    # Check valid judgement.
    if obj is None:
        return False

    if obj.get("judge_error") is not None:
        return False

    if obj.get("judge_score") not in {0, 1}:
        return False

    return True

def load_existing_jsonl(path: Path) -> Dict[str, Dict[str, Any]]:
    # Load latest rows.
    existing = {}

    if not path.exists():
        return existing

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            key = str(obj.get("_judge_key"))
            existing[key] = obj

    return existing

def append_jsonl(path: Path, obj: Dict[str, Any]):
    # Append JSONL row.
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def get_output_paths(task: Dict[str, Any]):
    # Get output paths.
    task_id = task["task_id"]

    out_jsonl = GDRIVE_OUTPUT_DIR / f"{task_id}__judged.jsonl"
    out_json = GDRIVE_OUTPUT_DIR / f"{task_id}__judged.json"

    return out_jsonl, out_json

def build_output_row(
    task: Dict[str, Any],
    row_id: int,
    key: str,
    record: Dict[str, Any],
    judge_result: Dict[str, Any],
) -> Dict[str, Any]:
    # Build output row.
    return {
        "_judge_key": key,
        "_row_id": row_id,
        "_task_id": task["task_id"],
        "_answer_model": task["answer_model"],
        "_dataset": task["dataset"],
        "_source_path": str(task["source_path"]),

        "source_index": record.get("source_index"),
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("gt"),
        "response": record.get("response"),

        "attempt": record.get("attempt"),
        "max_tokens_used": record.get("max_tokens_used"),
        "completion_tokens": record.get("completion_tokens"),
        "reasoning_tokens": record.get("reasoning_tokens"),
        "retry_reason": record.get("retry_reason"),

        **judge_result,
    }

def judge_task(task: Dict[str, Any]) -> Path:
    # Judge all rows in one task.
    task_id = task["task_id"]
    data = all_input_data[task_id]

    out_jsonl, out_json = get_output_paths(task)

    for round_id in range(1, TASK_MAX_ROUNDS + 1):
        existing = load_existing_jsonl(out_jsonl)

        todo = []

        for row_id, record in enumerate(data):
            key = make_result_key(record, row_id)
            current = existing.get(key)

            if not is_valid_judgement(current):
                todo.append((row_id, key, record))

        print("\nTask:", task_id)
        print("Answer model:", task["answer_model"])
        print("Dataset:", task["dataset"])
        print("Round:", round_id)
        print("Rows:", len(data))
        print("Valid already judged:", len(data) - len(todo))
        print("Remaining or failed:", len(todo))

        if not todo:
            break

        with ThreadPoolExecutor(max_workers=REQUEST_CONCURRENCY) as executor:
            futures = {
                executor.submit(judge_one_record, record): (row_id, key, record)
                for row_id, key, record in todo
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"Judging {task_id} | round {round_id}",
            ):
                row_id, key, record = futures[future]

                try:
                    judge_result = future.result()
                except Exception as e:
                    judge_result = {
                        "judge_reason": None,
                        "judge_score": None,
                        "judge_raw": None,
                        "judge_error": repr(e),
                        "judge_latency_sec": None,
                        "judge_method": "future_error",
                    }

                out_obj = build_output_row(
                    task=task,
                    row_id=row_id,
                    key=key,
                    record=record,
                    judge_result=judge_result,
                )

                append_jsonl(out_jsonl, out_obj)

    existing = load_existing_jsonl(out_jsonl)

    ordered = []
    missing_or_failed = 0

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)
        obj = existing.get(key)

        if not is_valid_judgement(obj):
            missing_or_failed += 1

        if obj is not None:
            ordered.append(obj)

    print("Remaining invalid after task rounds:", missing_or_failed)

    # Save preliminary JSON.
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)

    print("Saved JSONL:", out_jsonl)
    print("Saved preliminary JSON:", out_json)

    return out_json

In [ ]:
# 11
# Run judge for all tasks.
judged_json_paths = []

for task in JUDGE_TASKS:
    judged_path = judge_task(task)
    judged_json_paths.append(judged_path)

print("\nAll task rounds finished.")
print("Now run Cell 11.5 to repair and finalize.")

for path in judged_json_paths:
    print(path)


Task: qwen3_5__hotpotqa
Answer model: qwen3.5
Dataset: hotpotqa
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging qwen3_5__hotpotqa | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


Task: qwen3_5__hotpotqa
Answer model: qwen3.5
Dataset: hotpotqa
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Remaining invalid after task rounds: 0
Saved JSONL: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__hotpotqa__judged.jsonl
Saved preliminary JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__hotpotqa__judged.json

Task: qwen3_5__2wikimultihopqa
Answer model: qwen3.5
Dataset: 2wikimultihopqa
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging qwen3_5__2wikimultihopqa | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


Task: qwen3_5__2wikimultihopqa
Answer model: qwen3.5
Dataset: 2wikimultihopqa
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Remaining invalid after task rounds: 0
Saved JSONL: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__2wikimultihopqa__judged.jsonl
Saved preliminary JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__2wikimultihopqa__judged.json

Task: gemma4__hotpotqa
Answer model: gemma4
Dataset: hotpotqa
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


...


Task: gemma4__2wikimultihopqa
Answer model: gemma4
Dataset: 2wikimultihopqa
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Remaining invalid after task rounds: 0
Saved JSONL: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__2wikimultihopqa__judged.jsonl
Saved preliminary JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__2wikimultihopqa__judged.json

Task: gpt-oss-120b__hotpotqa
Answer model: gpt-oss-120b
Dataset: hotpotqa
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging gpt-oss-120b__hotpotqa | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


Task: gpt-oss-120b__hotpotqa
Answer model: gpt-oss-120b
Dataset: hotpotqa
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Remaining invalid after task rounds: 0
Saved JSONL: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__hotpotqa__judged.jsonl
Saved preliminary JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__hotpotqa__judged.json

Task: gpt-oss-120b__2wikimultihopqa
Answer model: gpt-oss-120b
Dataset: 2wikimultihopqa
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging gpt-oss-120b__2wikimultihopqa | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


Task: gpt-oss-120b__2wikimultihopqa
Answer model: gpt-oss-120b
Dataset: 2wikimultihopqa
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Remaining invalid after task rounds: 0
Saved JSONL: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__2wikimultihopqa__judged.jsonl
Saved preliminary JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__2wikimultihopqa__judged.json

All task rounds finished.
Now run Cell 11.5 to repair and finalize.
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__hotpotqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__2wikimultihopqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__hotpotqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__2wik

In [ ]:
# 12
# Repair and finalize all results.
from tqdm.auto import tqdm
from pathlib import Path
import json

def finalize_task(task: Dict[str, Any]) -> Path:
    # Repair and save final JSON.
    task_id = task["task_id"]
    data = all_input_data[task_id]

    out_jsonl, out_json = get_output_paths(task)
    latest = load_existing_jsonl(out_jsonl)

    failed_items = []

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)
        current = latest.get(key)

        if not is_valid_judgement(current):
            failed_items.append((row_id, key, record, current))

    print("\nTask:", task_id)
    print("Answer model:", task["answer_model"])
    print("Dataset:", task["dataset"])
    print("Failed or missing before repair:", len(failed_items))

    for row_id, key, record, old_obj in tqdm(failed_items, desc=f"Repairing {task_id}"):
        repair_result = repair_judge_one_record(record)

        repaired_obj = build_output_row(
            task=task,
            row_id=row_id,
            key=key,
            record=record,
            judge_result=repair_result,
        )

        repaired_obj["judge_repaired"] = True

        append_jsonl(out_jsonl, repaired_obj)
        latest[key] = repaired_obj

    ordered = []
    final_defaulted = 0

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)
        obj = latest.get(key)

        if not is_valid_judgement(obj):
            final_defaulted += 1

            fallback_result = {
                "judge_reason": "Final safety fallback; defaulted to incorrect.",
                "judge_score": 0,
                "judge_raw": None,
                "judge_error": None,
                "judge_latency_sec": None,
                "judge_method": "final_default_0",
                "judge_repaired": True,
            }

            obj = build_output_row(
                task=task,
                row_id=row_id,
                key=key,
                record=record,
                judge_result=fallback_result,
            )

            append_jsonl(out_jsonl, obj)
            latest[key] = obj

        ordered.append(obj)

    final_failed = [x for x in ordered if not is_valid_judgement(x)]

    print("Final rows:", len(ordered))
    print("Final invalid:", len(final_failed))
    print("Final defaulted to 0:", final_defaulted)

    assert len(ordered) == len(data), f"Row count mismatch in {task_id}"
    assert len(final_failed) == 0, f"Invalid judgements remain in {task_id}"

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)

    print("Saved final JSON:", out_json)

    return out_json

judged_json_paths = []

for task in JUDGE_TASKS:
    judged_path = finalize_task(task)
    judged_json_paths.append(judged_path)

print("\nRepair and finalization complete.")

for path in judged_json_paths:
    print(path)


Task: qwen3_5__hotpotqa
Answer model: qwen3.5
Dataset: hotpotqa
Failed or missing before repair: 0


Repairing qwen3_5__hotpotqa: 0it [00:00, ?it/s]

Final rows: 1000
Final invalid: 0
Final defaulted to 0: 0
Saved final JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__hotpotqa__judged.json

Task: qwen3_5__2wikimultihopqa
Answer model: qwen3.5
Dataset: 2wikimultihopqa
Failed or missing before repair: 0


Repairing qwen3_5__2wikimultihopqa: 0it [00:00, ?it/s]

Final rows: 1000
Final invalid: 0
Final defaulted to 0: 0
Saved final JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__2wikimultihopqa__judged.json

Task: gemma4__hotpotqa
Answer model: gemma4
Dataset: hotpotqa
Failed or missing before repair: 0


...

Final rows: 1000
Final invalid: 0
Final defaulted to 0: 0
Saved final JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__2wikimultihopqa__judged.json

Task: gpt-oss-120b__hotpotqa
Answer model: gpt-oss-120b
Dataset: hotpotqa
Failed or missing before repair: 0


Repairing gpt-oss-120b__hotpotqa: 0it [00:00, ?it/s]

Final rows: 1000
Final invalid: 0
Final defaulted to 0: 0
Saved final JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__hotpotqa__judged.json

Task: gpt-oss-120b__2wikimultihopqa
Answer model: gpt-oss-120b
Dataset: 2wikimultihopqa
Failed or missing before repair: 0


Repairing gpt-oss-120b__2wikimultihopqa: 0it [00:00, ?it/s]

Final rows: 1000
Final invalid: 0
Final defaulted to 0: 0
Saved final JSON: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__2wikimultihopqa__judged.json

Repair and finalization complete.
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__hotpotqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/qwen3_5__2wikimultihopqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__hotpotqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gemma4__2wikimultihopqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__hotpotqa__judged.json
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/llm_judge_qwen35_27b/gpt-oss-120b__2wikimultihopqa__judged.json


In [ ]:
# 13
# Load all final judged outputs.
import pandas as pd
import json

rows = []

for path in judged_json_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows.extend(data)

df = pd.DataFrame(rows)

df["judge_score"] = pd.to_numeric(df["judge_score"], errors="coerce")

print("Total rows:", len(df))
print("Tasks:", df["_task_id"].nunique())
print("Answer models:", df["_answer_model"].nunique())
print("Datasets:", df["_dataset"].nunique())
print("Invalid judgements:", df["judge_score"].isna().sum())

display(df.head())

Total rows: 6000
Tasks: 6
Answer models: 3
Datasets: 2
Invalid judgements: 0


,_judge_key,_row_id,_task_id,_answer_model,_dataset,_source_path,source_index,type,question,gt,...,max_tokens_used,completion_tokens,reasoning_tokens,retry_reason,judge_reason,judge_score,judge_raw,judge_error,judge_latency_sec,judge_method
0,row_id:0,0,qwen3_5__hotpotqa,qwen3.5,hotpotqa,/content/drive/MyDrive/final_project/baseline/...,None,comparison,Where operation Operation Dragoon and Battle o...,yes,...,None,None,None,None,The candidate correctly identifies the two dif...,1,"{\n ""reason"": ""The candidate correctly identi...",None,24.480717,guided_json
1,row_id:1,1,qwen3_5__hotpotqa,qwen3.5,hotpotqa,/content/drive/MyDrive/final_project/baseline/...,None,bridge,"What country does Eric A. Sykes and Eccles, Gr...",England,...,None,None,None,None,The candidate response matches the ground-trut...,1,"{\n ""reason"": ""The candidate response matches...",None,23.777534,guided_json
2,row_id:2,2,qwen3_5__hotpotqa,qwen3.5,hotpotqa,/content/drive/MyDrive/final_project/baseline/...,None,comparison,"Which is a Macedonian weekly, Hänt Extra or Te...",Tea Moderna,...,None,None,None,None,The candidate correctly identifies Tea Moderna...,1,"{\n ""reason"": ""The candidate correctly identi...",None,25.676599,guided_json
3,row_id:3,3,qwen3_5__hotpotqa,qwen3.5,hotpotqa,/content/drive/MyDrive/final_project/baseline/...,None,bridge,Conrad Anker located the body of a mountaineer...,Mount Everest,...,None,None,None,None,The candidate response matches the ground trut...,1,"{\n ""reason"": ""The candidate response matches...",None,24.110239,guided_json
4,row_id:4,4,qwen3_5__hotpotqa,qwen3.5,hotpotqa,/content/drive/MyDrive/final_project/baseline/...,None,comparison,"Which publication publishes more frequently, U...",The New York Enterprise Report,...,None,None,None,None,The candidate response matches the ground trut...,1,"{\n ""reason"": ""The candidate response matches...",None,24.349110,guided_json


In [ ]:
# 14
# Compute accuracy tables.
valid_df = df[df["judge_score"].isin([0, 1])].copy()

if len(valid_df) != len(df):
    raise RuntimeError("Some judgements are invalid. Run Cell 11.5 again.")

def accuracy_table(dataframe, group_cols):
    # Build accuracy table.
    result = (
        dataframe
        .groupby(group_cols, dropna=False)
        .agg(
            total=("judge_score", "size"),
            correct=("judge_score", "sum"),
            accuracy=("judge_score", "mean"),
        )
        .reset_index()
    )

    result["correct"] = result["correct"].astype(int)
    result["accuracy_percent"] = result["accuracy"] * 100.0

    return result.sort_values(group_cols).reset_index(drop=True)

per_model_dataset = accuracy_table(valid_df, ["_answer_model", "_dataset"])
per_model_dataset_type = accuracy_table(valid_df, ["_answer_model", "_dataset", "type"])
per_model_overall = accuracy_table(valid_df, ["_answer_model"])
per_dataset_overall = accuracy_table(valid_df, ["_dataset"])
combined_type = accuracy_table(valid_df, ["type"])

combined_overall = pd.DataFrame([
    {
        "scope": "all_tasks_combined",
        "total": int(valid_df["judge_score"].size),
        "correct": int(valid_df["judge_score"].sum()),
        "accuracy": float(valid_df["judge_score"].mean()),
        "accuracy_percent": float(valid_df["judge_score"].mean() * 100.0),
    }
])

print("Per model and dataset accuracy:")
display(per_model_dataset)

print("Per model, dataset, and type accuracy:")
display(per_model_dataset_type)

print("Per model overall accuracy:")
display(per_model_overall)

print("Per dataset overall accuracy:")
display(per_dataset_overall)

print("Combined per-type accuracy:")
display(combined_type)

print("Combined overall accuracy:")
display(combined_overall)

Per model and dataset accuracy:


,_answer_model,_dataset,total,correct,accuracy,accuracy_percent
0,gemma4,2wikimultihopqa,1000,251,0.251,25.1
1,gemma4,hotpotqa,1000,441,0.441,44.1
2,gpt-oss-120b,2wikimultihopqa,1000,322,0.322,32.2
3,gpt-oss-120b,hotpotqa,1000,539,0.539,53.9
4,qwen3.5,2wikimultihopqa,1000,352,0.352,35.2
5,qwen3.5,hotpotqa,1000,564,0.564,56.4


Per model, dataset, and type accuracy:


,_answer_model,_dataset,type,total,correct,accuracy,accuracy_percent
0,gemma4,2wikimultihopqa,bridge_comparison,250,4,0.016000,1.600000
1,gemma4,2wikimultihopqa,comparison,250,128,0.512000,51.200000
2,gemma4,2wikimultihopqa,compositional,250,37,0.148000,14.800000
3,gemma4,2wikimultihopqa,inference,250,82,0.328000,32.800000
4,gemma4,hotpotqa,bridge,700,311,0.444286,44.428571
5,gemma4,hotpotqa,comparison,300,130,0.433333,43.333333
6,gpt-oss-120b,2wikimultihopqa,bridge_comparison,250,56,0.224000,22.400000
7,gpt-oss-120b,2wikimultihopqa,comparison,250,133,0.532000,53.200000
8,gpt-oss-120b,2wikimultihopqa,compositional,250,45,0.180000,18.000000
9,gpt-oss-120b,2wikimultihopqa,inference,250,88,0.352000,35.200000


Per model overall accuracy:


...

,_dataset,total,correct,accuracy,accuracy_percent
0,2wikimultihopqa,3000,925,0.308333,30.833333
1,hotpotqa,3000,1544,0.514667,51.466667


Combined per-type accuracy:


,type,total,correct,accuracy,accuracy_percent
0,bridge,2100,1054,0.501905,50.190476
1,bridge_comparison,750,154,0.205333,20.533333
2,comparison,1650,910,0.551515,55.151515
3,compositional,750,133,0.177333,17.733333
4,inference,750,218,0.290667,29.066667


Combined overall accuracy:


,scope,total,correct,accuracy,accuracy_percent
0,all_tasks_combined,6000,2469,0.4115,41.15


In [ ]:
# Show overall accuracy per model and dataset.
model_dataset_overall = (
    valid_df
    .groupby(["_answer_model", "_dataset"], dropna=False)
    .agg(
        total=("judge_score", "size"),
        correct=("judge_score", "sum"),
        accuracy=("judge_score", "mean"),
    )
    .reset_index()
)

model_dataset_overall["correct"] = model_dataset_overall["correct"].astype(int)
model_dataset_overall["accuracy_percent"] = model_dataset_overall["accuracy"] * 100.0

model_dataset_overall = model_dataset_overall.sort_values(
    ["_answer_model", "_dataset"]
).reset_index(drop=True)

print("Overall accuracy for each model on each dataset:")
display(model_dataset_overall[
    [
        "_answer_model",
        "_dataset",
        "total",
        "correct",
        "accuracy_percent",
    ]
])

# Pivot table for easier comparison.
model_dataset_percent_table = model_dataset_overall.pivot(
    index="_answer_model",
    columns="_dataset",
    values="accuracy_percent",
).reset_index()

print("Accuracy percent table:")
display(model_dataset_percent_table)

Overall accuracy for each model on each dataset:


,_answer_model,_dataset,total,correct,accuracy_percent
0,gemma4,2wikimultihopqa,1000,251,25.1
1,gemma4,hotpotqa,1000,441,44.1
2,gpt-oss-120b,2wikimultihopqa,1000,322,32.2
3,gpt-oss-120b,hotpotqa,1000,539,53.9
4,qwen3.5,2wikimultihopqa,1000,352,35.2
5,qwen3.5,hotpotqa,1000,564,56.4


Accuracy percent table:


_dataset,_answer_model,2wikimultihopqa,hotpotqa
0,gemma4,25.1,44.1
1,gpt-oss-120b,32.2,53.9
2,qwen3.5,35.2,56.4
